In [ ]:
class Icd9Encoder(BaseEstimator, TransformerMixin):

    def __init__(self, comorbidity_df=None):
        self.comorbidity_df = comorbidity_df  # safe, sklearn cloneable

    def fit(self, X, y):
        df = X.copy().reset_index(drop = True)
        y = y.reset_index(drop = True)
        df['target'] = y.values

        com = self.comorbidity_df   # ACCESS HERE SAFELY

        merged = df[['subject_id','hadm_id','target']].merge(
            com, on=['subject_id','hadm_id'], how='left'
        )

        self.icd_mortality_ = merged.groupby('icd9_code')['target'].mean()
        self.global_mean_ = df['target'].mean()
        return self

    def transform(self, X):
        df = X.copy().reset_index(drop = True)
        com = self.comorbidity_df

        merged = df[['subject_id','hadm_id']].merge(
            com, on=['subject_id','hadm_id'], how='left'
        )

        merged['mortality_proxy'] = merged['icd9_code'].map(self.icd_mortality_)
        merged['mortality_proxy'] = merged['mortality_proxy'].fillna(self.global_mean_)

        agg = merged.groupby(['subject_id','hadm_id']).agg(
            max_mortality=('mortality_proxy','max'),
            mean_mortality=('mortality_proxy','mean'),
            count_comorbidities=('icd9_code','count')
        ).reset_index()

        agg = agg.fillna({
            'max_mortality': self.global_mean_,
            'mean_mortality': self.global_mean_,
            'count_comorbidities': 0
        })

        return df.merge(agg, on=['subject_id','hadm_id'], how='left').reset_index(drop = True)

class IndexResetter(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        return X.reset_index(drop=True)


In [5]:
! source "../.venv/bin/activate"


In [ ]:
train.iloc[num]